# 12 Views and Copies

| | |
|---|---|
| **Path** | NumPy |
| **Prerequisite** | 11 |

A view is a second way to look at the same memory. A copy is a new block. Editing a view edits the original. Editing a copy does not.

Slices are views. Fancy indexes and boolean masks are copies. `reshape` is a view when the elements are already contiguous. `.copy()` is how you force a new block.

`np.shares_memory(a, b)` answers whether the two arrays can see the same bytes.


## Example 1 — A sliced row is the original row

### Predict

`row = grid[0]` does not copy. Setting `row[0] = 99` writes into `grid[0, 0]`. `shares_memory` is true.

### Run


In [1]:
# Example 1 — A sliced row is the original row.
import numpy as np

grid = np.arange(6).reshape(2, 3)
row = grid[0]
print(np.shares_memory(grid, row))
row[0] = 99
print(grid)


True
[[99  1  2]
 [ 3  4  5]]


## Example 2 — A list of rows is a copy

### Predict

`grid[[0]]` uses a fancy index. It is a copy, so setting its first entry to 7 leaves `grid` unchanged. After example 1 the corner is 99, and it stays 99.

### Run


In [2]:
# Example 2 — A list of rows is a copy.
import numpy as np

grid = np.arange(6).reshape(2, 3)
picked = grid[[0]]
print(np.shares_memory(grid, picked))
picked[0, 0] = 7
print(grid[0, 0])
print(picked[0, 0])


False
0
7


## Example 3 — copy protects the parent

### Predict

`grid[0].copy()` is a new row. Assigning 5 into that copy does not change `grid[0, 1]`, which stays 1.

### Run


In [3]:
# Example 3 — copy protects the parent.
import numpy as np

grid = np.arange(6).reshape(2, 3)
safe = grid[0].copy()
safe[1] = 5
print(np.shares_memory(grid, safe))
print(grid[0, 1])
print(safe[1])


False
1
5


## Example 4 — A mask is a copy, so assign through the mask instead

### Predict

`positives = values[values > 0]` is a copy. Multiplying that copy by 10 does not change `values`. The assignment `values[values > 0] *= 10` writes through the mask into the original, so `[1, -2, 3]` becomes `[10, -2, 30]`.

### Run


In [4]:
import numpy as np

values = np.array([1, -2, 3])
positives = values[values > 0]
positives *= 10
# The copy changed. The original did not.
print(values)
values[values > 0] *= 10
print(values)


[ 1 -2  3]
[10 -2 30]


## A pitfall

A view can outlive the meaning you had in mind. A function that returns `grid[0]` returns a door into the caller's data. Return `.copy()` when the caller should be free to edit the result.

### Run


In [5]:
import numpy as np

def first_row(grid):
    # The caller receives a view, not a private row.
    return grid[0]

grid = np.arange(6).reshape(2, 3)
borrowed = first_row(grid)
borrowed[0] = 8
print(grid[0, 0])


8


## Summary

Slices share memory with the parent. Fancy indexes and boolean masks copy. `.copy()` is the explicit new block. Assign through a mask when you mean to edit the original; editing the array you got back from a mask edits only the copy. If a function should not expose the parent's memory, return a copy.
